# 3. A scoring rate that changes with time

Notebook 2 rejected a constant scoring rate. This notebook fits models where the rate depends on the minute, compares them, and then checks whether the best one is adequate.

**The model.** Match *m* has scoring rate `c_m * s(t)`: its own level `c_m`, times a shape `s(t)` shared by all matches. This is a *non-homogeneous Poisson process*.

**Why we can ignore the levels.** Given the number of goals in a match, the goal times are independent draws from the density `f(t) = s(t) / (integral of s over 0 to 90)`. The levels `c_m` cancel. So fitting the shape is the same as fitting a probability density to the pooled goal times, and the differences between matches found in notebook 1 do not get in the way.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import scipy.stats as st
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt
from load import load

matches, goals = load("../data/worldcup.json")

T = 90.0
reg = goals[~goals.in_extra_time]
clean = reg[reg.stoppage == 0].copy()       # nominal minutes only, as in notebook 2

# spread each goal uniformly over its minute; same seed as notebook 2
rng = np.random.default_rng(0)
clean["t"] = clean.minute.to_numpy() - rng.uniform(0, 1, len(clean))

t = clean.t.to_numpy()
N = len(t)
print(f"{N} goals from {len(matches)} matches")

## 3.1 Three candidate shapes

| Model | Shape `s(t)` | Extra parameters |
| --- | --- | --- |
| Constant | 1 | 0 |
| Log-linear | `exp(b * t)`: the rate grows by the same percentage every minute | 1 |
| Half-time step | one rate before half-time, another after | 1 |
| Six periods | a separate rate for each 15-minute period | 5 |

Each function below returns the log-likelihood of the goal times under that density.

In [ ]:
def loglik_constant(t):
    return -len(t) * np.log(T)


def loglik_loglinear(b, t):
    # density: b * exp(b t) / (exp(b T) - 1)
    return np.sum(np.log(b) + b * t - np.log(np.expm1(b * T)))


def fit_loglinear(t):
    res = minimize_scalar(lambda b: -loglik_loglinear(b, t), bounds=(1e-6, 0.1), method="bounded")
    return res.x, -res.fun


def fit_pieces(t, edges):
    # piecewise-constant density; the MLE of each piece's probability is its share of the goals
    counts, _ = np.histogram(t, edges)
    widths = np.diff(edges)
    probs = counts / counts.sum()
    return probs, np.sum(counts * np.log(probs / widths))

## 3.2 Fit the log-linear model

The estimate of `b` is found by maximising the log-likelihood numerically. Its standard error comes from the curvature of the log-likelihood at the maximum: a sharply peaked likelihood means a precise estimate.

`exp(90 * b)` is the easiest number to read: how many times higher the scoring rate is at minute 90 than at kick-off.

In [ ]:
b_hat, ll_loglin = fit_loglinear(t)

# standard error from the second derivative of the log-likelihood (observed information)
h = 1e-5
curvature = (loglik_loglinear(b_hat + h, t) - 2 * loglik_loglinear(b_hat, t)
             + loglik_loglinear(b_hat - h, t)) / h**2
se = 1 / np.sqrt(-curvature)

lo, hi = b_hat - 1.96 * se, b_hat + 1.96 * se
print(f"b = {b_hat:.5f} per minute (standard error {se:.5f})")
print(f"rate at minute 90 relative to kick-off: {np.exp(T * b_hat):.2f}"
      f"   95% interval: {np.exp(T * lo):.2f} to {np.exp(T * hi):.2f}")

## 3.3 Compare the models

Two tools:

- **Likelihood-ratio test.** Twice the gain in log-likelihood over the constant model is compared with a chi-square distribution whose degrees of freedom are the number of extra parameters.
- **AIC** = `-2 * log-likelihood + 2 * parameters`. Lower is better. It penalises extra parameters, so a flexible model has to earn them.

In [ ]:
ll_const = loglik_constant(t)
p_half, ll_half = fit_pieces(t, np.array([0, 45, 90.0]))
p_six, ll_six = fit_pieces(t, np.arange(0, 91, 15.0))

rows = []
for name, ll, k in [("constant", ll_const, 0), ("log-linear", ll_loglin, 1),
                    ("half-time step", ll_half, 1), ("six periods", ll_six, 5)]:
    lr = 2 * (ll - ll_const)
    rows.append({"model": name, "extra params": k, "log-lik": round(ll, 2),
                 "AIC": round(-2 * ll + 2 * k, 2),
                 "LR vs constant": round(lr, 2),
                 "p": round(st.chi2.sf(lr, k), 5) if k else None})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

In [ ]:
# observed goals per match in each 15-minute period, with the fitted shapes over them
M = len(matches)
edges = np.arange(0, 91, 15.0)
counts, _ = np.histogram(t, edges)
grid = np.linspace(0, T, 400)

density_loglin = b_hat * np.exp(b_hat * grid) / np.expm1(b_hat * T)
density_half = np.where(grid <= 45, p_half[0] / 45, p_half[1] / 45)
scale = N / M * 15                      # density -> goals per match per 15 minutes

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(edges[:-1] + 7.5, counts / M, width=13, color="#9bb7d4", label="observed")
ax.plot(grid, np.full_like(grid, 1 / T) * scale, "--", color="grey", label="constant")
ax.plot(grid, density_loglin * scale, color="#1f3b5c", label="log-linear")
ax.plot(grid, density_half * scale, color="#e0a458", label="half-time step")
ax.set_xticks(edges)
ax.set_xlabel("minute of the match")
ax.set_ylabel("goals per match per 15 minutes")
ax.set_title("Fitted scoring-rate shapes")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig("../figures/03_fitted_rates.png", dpi=150)
plt.show()

## 3.4 Is the fitted model adequate? The time-rescaling check

Beating the constant model does not show that a model is *right*. The time-rescaling theorem gives a direct check.

**The theorem.** Take a Poisson process with rate `r(t)` and run the clock at speed `r(t)`: replace each event time `t` with the cumulative rate up to `t`. On the new clock the events form a Poisson process with constant rate 1. A changing rate is the same as a constant rate on a stretched clock.

**How it is used here.** Push each goal time through the fitted cumulative shape `F(t)`, which runs from 0 at kick-off to 1 at minute 90. If the fitted shape is right, the rescaled times are uniform on (0, 1), by the same conditioning argument as notebook 2. So we rerun the uniformity test on the rescaled clock. A small p-value would mean the shape is wrong.

In [ ]:
def cdf_loglinear(t, b):
    return np.expm1(b * t) / np.expm1(b * T)


def cdf_half(t, p_first):
    return np.where(t <= 45, p_first * t / 45, p_first + (1 - p_first) * (t - 45) / 45)


rescaled = {
    "constant": t / T,
    "log-linear": cdf_loglinear(t, b_hat),
    "half-time step": cdf_half(t, p_half[0]),
}

fig, ax = plt.subplots(figsize=(6, 4))
for (name, u), colour in zip(rescaled.items(), ["grey", "#1f3b5c", "#e0a458"]):
    ks = st.kstest(u, "uniform")
    print(f"{name:<15} KS statistic = {ks.statistic:.3f}, p = {ks.pvalue:.4f}")
    us = np.sort(u)
    ax.step(us, np.arange(1, N + 1) / N - us, color=colour, label=name)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("rescaled time")
ax.set_ylabel("empirical CDF minus uniform CDF")
ax.set_title("Time-rescaling check: closer to zero is better")
ax.legend()
fig.tight_layout()
fig.savefig("../figures/03_time_rescaling.png", dpi=150)
plt.show()

## 3.5 Does it hold on a tournament the model has not seen?

Everything above was fitted and checked on the same goals. A fairer test: fit the shapes on 2014, 2018 and 2022, then apply the time-rescaling check to 2026 without refitting.

In [ ]:
train = clean[clean.year < 2026].t.to_numpy()
test = clean[clean.year == 2026].t.to_numpy()
print(f"fitted on {len(train)} goals (2014 to 2022), tested on {len(test)} goals (2026)\n")

b_train, _ = fit_loglinear(train)
p_train, _ = fit_pieces(train, np.array([0, 45, 90.0]))

for name, u in [("constant", test / T),
                ("log-linear", cdf_loglinear(test, b_train)),
                ("half-time step", cdf_half(test, p_train[0]))]:
    ks = st.kstest(u, "uniform")
    print(f"{name:<15} KS statistic = {ks.statistic:.3f}, p = {ks.pvalue:.3f}")

## Conclusion

A scoring rate that rises through the match fits the goal times well. Under the log-linear model the rate grows by about 0.5% per minute, so it is about 1.6 times higher at minute 90 than at kick-off (95% interval 1.2 to 2.0). That model beats a constant rate clearly (likelihood ratio 12.05 on 1 degree of freedom, p = 0.0005).

The data cannot tell a smooth rise from a jump at half-time. The half-time step model fits almost identically (AIC 6261.2 against 6262.7), and a separate rate for each 15-minute period does not earn its four extra parameters (AIC 6263.6). With about 700 goals, both one-parameter descriptions are consistent with the data.

The time-rescaling check supports both. On the constant-rate clock the goal times are far from uniform (p = 0.0003); on the log-linear clock and the half-time clock they are consistent with uniform (p = 0.68 and p = 0.55). So a non-homogeneous Poisson process with one shape parameter is an adequate description of when goals are scored.

Out of sample the picture is the same but weaker. With the shapes fitted on 2014 to 2022, the 2026 goals are consistent with both time-varying models (p = 0.33 and p = 0.60). They are not clearly inconsistent with a constant rate either (p = 0.09): 254 goals from one tournament are too few to reject it alone.

*Check every number against the output above, then put this in your own words.*